In [ ]:
# ============================================================
# ScamShield AI — Notebook 4: Image/Screenshot Analyzer
# ============================================================
# MODULE 3: Screenshot → Text → Classification
#
# PIPELINE:
# Screenshot → Preprocess → EasyOCR → Extracted Text
# → Text Classifier → Scam Probability
#
# WHY THIS APPROACH?
# Many scams come as screenshots. WhatsApp forwards,
# bank SMS screenshots, fake offer images.
# OCR extracts the text, then our text model classifies it.
# ============================================================

import os
import re
import numpy as np
import cv2
import easyocr
from PIL import Image, ImageFilter, ImageEnhance
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

print("=" * 55)
print("  ScamShield AI — Image OCR Analyzer")
print("=" * 55)
print()
print("Initializing EasyOCR...")
print("(First run downloads language models ~100MB)")
print()

# Initialize EasyOCR reader
# ['en'] = English language
# gpu=False = use CPU (set True if you have GPU)
reader = easyocr.Reader(['en', 'hi'], gpu=False)
print("EasyOCR ready!")
print("  Supported languages: English + Hindi")


# ============================================================
# IMAGE PREPROCESSING PIPELINE
# ============================================================
# Raw screenshots are noisy. Preprocessing improves OCR accuracy.
# Steps:
# 1. Convert to grayscale (OCR doesn't need color)
# 2. Resize (upscale small images)
# 3. Denoise (remove noise/grain)
# 4. Enhance contrast (make text stand out)
# 5. Binarize (convert to pure black/white)

def preprocess_image_for_ocr(image_input):
    """
    Preprocess an image for optimal OCR accuracy.
    
    Args:
        image_input: PIL Image, numpy array, or file path string
        
    Returns:
        numpy array: Preprocessed grayscale image ready for OCR
        PIL Image: Visual of processed image for display
    """
    
    # ── Step 1: Load image ────────────────────────────────
    if isinstance(image_input, str):
        # File path provided
        pil_image = Image.open(image_input)
    elif isinstance(image_input, np.ndarray):
        # NumPy array (OpenCV format)
        pil_image = Image.fromarray(image_input)
    else:
        # PIL Image
        pil_image = image_input
    
    # ── Step 2: Convert to RGB (handles RGBA/grayscale) ──
    pil_image = pil_image.convert('RGB')
    
    # ── Step 3: Upscale small images ──────────────────────
    # OCR works better on larger images
    # Min 300x300, max 2000px on longest side
    w, h = pil_image.size
    min_size = 300
    
    if w < min_size or h < min_size:
        scale = max(min_size/w, min_size/h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        pil_image = pil_image.resize((new_w, new_h), Image.LANCZOS)
    
    # ── Step 4: Enhance contrast ──────────────────────────
    enhancer = ImageEnhance.Contrast(pil_image)
    pil_image = enhancer.enhance(1.5)  # 1.5x contrast boost
    
    # ── Step 5: Sharpen ───────────────────────────────────
    pil_image = pil_image.filter(ImageFilter.SHARPEN)
    
    # ── Step 6: Convert to NumPy for OpenCV ───────────────
    img_array = np.array(pil_image)
    
    # ── Step 7: Convert to grayscale ──────────────────────
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    
    # ── Step 8: Denoise ───────────────────────────────────
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    
    # ── Step 9: Adaptive thresholding ─────────────────────
    # Better than simple threshold for varying lighting
    binary = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )
    
    # Convert back to PIL for display
    processed_pil = Image.fromarray(binary)
    
    return binary, processed_pil


def extract_text_from_image(image_input, reader):
    """
    Full pipeline: Image → Preprocessed → OCR → Clean text
    
    Args:
        image_input: PIL Image, numpy array, or file path
        reader: EasyOCR reader instance
        
    Returns:
        dict: {
            'text': full extracted text,
            'confidence': average confidence score,
            'word_count': number of words found,
            'raw_results': full EasyOCR output with bounding boxes
        }
    """
    
    # Preprocess
    processed_img, processed_pil = preprocess_image_for_ocr(image_input)
    
    # Run OCR
    # detail=1 returns bounding boxes + text + confidence
    ocr_results = reader.readtext(
        processed_img,
        detail=1,              # Include bounding boxes
        paragraph=False,       # Don't merge into paragraphs (better for SMS)
        min_size=10,           # Ignore very tiny text
        text_threshold=0.7,    # Minimum confidence to include text
        low_text=0.4,
        link_threshold=0.4,
    )
    
    if not ocr_results:
        return {
            'text': '',
            'confidence': 0.0,
            'word_count': 0,
            'raw_results': []
        }
    
    # Extract text and confidences
    texts = []
    confidences = []
    
    for (bbox, text, confidence) in ocr_results:
        if confidence >= 0.5:  # Only include confident predictions
            texts.append(text.strip())
            confidences.append(confidence)
    
    # Join all text pieces
    full_text = ' '.join(texts)
    
    # Clean the extracted text
    full_text = re.sub(r'\s+', ' ', full_text).strip()
    
    return {
        'text': full_text,
        'confidence': float(np.mean(confidences)) if confidences else 0.0,
        'word_count': len(full_text.split()),
        'raw_results': ocr_results
    }


print()
print("Image preprocessing pipeline ready!")
print("OCR extraction function ready!")

In [ ]:
# ============================================================
# CREATE TEST IMAGES PROGRAMMATICALLY
# ============================================================
# Since we don't have real scam screenshots,
# we create test images with scam text drawn on them.
# This validates our OCR pipeline works correctly.
# ============================================================

from PIL import ImageDraw, ImageFont

def create_test_scam_image(text, filename, bg_color=(255,255,255), 
                            text_color=(0,0,0)):
    """
    Create a test image that simulates an SMS screenshot.
    Used to validate our OCR pipeline.
    """
    
    # Create image
    width, height = 400, 200
    img = Image.new('RGB', (width, height), color=bg_color)
    draw = ImageDraw.Draw(img)
    
    # Draw a simple phone UI border
    draw.rectangle([10, 10, width-10, height-10], 
                   outline=(200, 200, 200), width=2)
    
    # Draw "SMS" header
    draw.rectangle([10, 10, width-10, 40], fill=(70, 130, 180))
    draw.text((20, 20), "SMS Message", fill=(255, 255, 255))
    
    # Draw the scam text (word-wrapped)
    words = text.split()
    lines = []
    current_line = []
    
    for word in words:
        current_line.append(word)
        if len(' '.join(current_line)) > 45:
            lines.append(' '.join(current_line[:-1]))
            current_line = [word]
    if current_line:
        lines.append(' '.join(current_line))
    
    y_position = 55
    for line in lines:
        draw.text((20, y_position), line, fill=text_color)
        y_position += 20
    
    # Save
    NOTEBOOK_DIR = os.getcwd()
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
    test_img_dir = os.path.join(PROJECT_ROOT, 'data', 'test_images')
    os.makedirs(test_img_dir, exist_ok=True)
    
    img_path = os.path.join(test_img_dir, filename)
    img.save(img_path)
    return img_path, img


# Create test images
print("Creating test images for OCR validation...")
print()

test_cases = [
    {
        'text': 'URGENT: Your SBI account will be blocked! Click http://sbi-kyc.xyz to verify OTP now!',
        'filename': 'test_scam_sbi.png',
        'expected': 'scam'
    },
    {
        'text': 'Your Amazon order #12345 has been shipped. Delivery by Jan 20.',
        'filename': 'test_legit_amazon.png', 
        'expected': 'legitimate'
    },
    {
        'text': 'Congratulations! You won Rs 50,000 in KBC Lucky Draw. Call 9876543210 to claim prize!',
        'filename': 'test_scam_kbc.png',
        'expected': 'scam'
    },
]

for tc in test_cases:
    path, img = create_test_scam_image(tc['text'], tc['filename'])
    print(f"Created: {tc['filename']} (Expected: {tc['expected']})")

print()
print("Running OCR on test images...")
print("=" * 55)

NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
test_img_dir = os.path.join(PROJECT_ROOT, 'data', 'test_images')

fig, axes = plt.subplots(len(test_cases), 2, figsize=(14, 4*len(test_cases)))
fig.suptitle('OCR Pipeline Test Results', fontsize=14, fontweight='bold')

ocr_results_list = []

for i, tc in enumerate(test_cases):
    img_path = os.path.join(test_img_dir, tc['filename'])
    pil_img = Image.open(img_path)
    
    # Run OCR
    result = extract_text_from_image(pil_img, reader)
    
    print(f"\nTest {i+1}: {tc['filename']}")
    print(f"  Expected:   {tc['expected']}")
    print(f"  Original:   {tc['text'][:60]}...")
    print(f"  Extracted:  {result['text'][:60]}...")
    print(f"  Confidence: {result['confidence']:.3f}")
    print(f"  Word count: {result['word_count']}")
    
    # Show original image
    axes[i][0].imshow(pil_img)
    axes[i][0].set_title(f"Original: {tc['filename']}")
    axes[i][0].axis('off')
    
    # Show processed image
    processed, _ = preprocess_image_for_ocr(pil_img)
    axes[i][1].imshow(processed, cmap='gray')
    axes[i][1].set_title(f"Processed + OCR: '{result['text'][:40]}...'")
    axes[i][1].axis('off')
    
    ocr_results_list.append(result)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SAVE OCR CONFIGURATION
# ============================================================
# The OCR itself doesn't need "saving" — EasyOCR loads its
# own models. We save the configuration and test results.
# ============================================================

import json

ocr_config = {
    'languages': ['en', 'hi'],
    'gpu': False,
    'text_threshold': 0.7,
    'min_confidence': 0.5,
    'preprocessing': {
        'contrast_factor': 1.5,
        'denoise': True,
        'adaptive_threshold': True
    }
}

NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
MODELS_DIR = os.path.join(PROJECT_ROOT, 'backend', 'saved_models')

config_path = os.path.join(MODELS_DIR, 'ocr_config.json')
with open(config_path, 'w') as f:
    json.dump(ocr_config, f, indent=2)

print(f"OCR configuration saved: {config_path}")
print()
print("OCR Pipeline Summary:")
print(f"  Engine:       EasyOCR")
print(f"  Languages:    English + Hindi")
print(f"  Preprocessing: Contrast + Denoise + Adaptive Threshold")
print(f"  Output:        Clean text → Text Classifier")
print()
print("The complete image pipeline is:")
print("  Screenshot (.png/.jpg)")
print("      ↓")
print("  preprocess_image_for_ocr()")
print("      ↓")
print("  EasyOCR reader.readtext()")
print("      ↓") 
print("  Extracted text")
print("      ↓")
print("  Text Classifier (Module 1)")
print("      ↓")
print("  Scam Probability Score")